# Pneumonia Detection & Localization — Member 3
## Grad-CAM Localization, IoU Evaluation & Streamlit Demo

**Member 3 responsibilities**
- `rc/src/gradcam.py` — Grad-CAM heatmaps from Member 2's DenseNet121 checkpoint.
- `rc/src/localization.py` — heatmap → approximate bounding boxes, IoU against RSNA ground truth (`all_annotations.csv`).
- `rc/app/app.py` — Streamlit demo: upload an X-ray → probability, heatmap, boxes.

**Protocol:** choose the heatmap threshold on `val.csv`, then evaluate `test.csv` **once** with that threshold.

> Grad-CAM is a visual explanation, not a trained detector. The boxes are approximate and the demo is not a clinical diagnosis.

### Step 1: GPU runtime
**Runtime → Change runtime type → T4 GPU**. Grad-CAM over the val/test positives takes a few minutes on GPU.

In [ ]:
!nvidia-smi

### Step 2: Mount Google Drive
The `dataset` and `Pneumonia_Project` folders are shared with you. In Drive, right-click each one under
**Shared with me → Organize → Add shortcut → My Drive** so they appear under `/content/drive/MyDrive/`.
The cell below also searches the shortcut locations automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, os

def find_dir(*names, must_contain=None):
    for name in names:
        candidates = ([f'/content/drive/MyDrive/{name}'] +
                      glob.glob(f'/content/drive/.shortcut-targets-by-id/*/{name}') +
                      glob.glob(f'/content/drive/MyDrive/*/{name}'))
        for c in candidates:
            if os.path.isdir(c) and (must_contain is None or os.path.exists(f'{c}/{must_contain}')):
                return c
    raise FileNotFoundError(f"Could not find any of {names}. Add a shortcut to the shared folder in My Drive.")

DATASET_DIR = find_dir('dataset', must_contain='stage_2_train_images.zip')
# The shared metadata folder is called 'metadata' (Pneumonia_Project/metadata in the owner's Drive)
METADATA_DIR = find_dir('Pneumonia_Project/metadata', 'metadata', must_contain='all_annotations.csv')

# best_model.pth from Member 2: searched in Pneumonia_Project/checkpoint(s)/ (own Drive or a shortcut
# to the shared folder). To use a specific file instead, set CHECKPOINT to its path.
ckpts = sorted(set(
    glob.glob('/content/drive/MyDrive/Pneumonia_Project/checkpoint*/best_model.pth') +
    glob.glob('/content/drive/MyDrive/*/Pneumonia_Project/checkpoint*/best_model.pth') +
    glob.glob('/content/drive/.shortcut-targets-by-id/*/Pneumonia_Project/checkpoint*/best_model.pth') +
    glob.glob('/content/drive/.shortcut-targets-by-id/*/checkpoint*/best_model.pth')), key=os.path.getmtime)
CHECKPOINT = ckpts[-1] if ckpts else '/content/drive/MyDrive/Pneumonia_Project/checkpoint/best_model.pth'
if len(ckpts) > 1:
    print('Several checkpoints found, using the newest:', *ckpts, sep='\n  ')
RESULTS_DIR = '/content/drive/MyDrive/Pneumonia_Project/results/localization'
IMG_DIR = '/content/rsna_train_images/stage_2_train_images'
os.makedirs(RESULTS_DIR, exist_ok=True)

os.environ.update(DATASET_DIR=DATASET_DIR, METADATA_DIR=METADATA_DIR, CHECKPOINT=CHECKPOINT,
                  RESULTS_DIR=RESULTS_DIR, IMG_DIR=IMG_DIR)
print('Dataset   :', DATASET_DIR, os.listdir(DATASET_DIR))
print('Metadata  :', METADATA_DIR, os.listdir(METADATA_DIR))
print('Checkpoint:', CHECKPOINT, 'OK' if os.path.exists(CHECKPOINT) else 'MISSING')

### Step 3: Clone the repository and switch to `feature/localization`

In [ ]:
%cd /content
!rm -rf Pneumonia_Detection_Localization
!git clone https://github.com/Tanujn06/Pneumonia_Detection_Localization.git
%cd /content/Pneumonia_Detection_Localization
!git switch feature/localization
!git log --oneline -3

### Step 4: Install dependencies and extract the training DICOMs to local SSD

In [ ]:
!pip install -q pydicom streamlit

import os
if not os.path.isdir(IMG_DIR) or len(os.listdir(IMG_DIR)) < 26000:
    print('Extracting stage_2_train_images.zip (a few minutes)...')
    !mkdir -p /content/rsna_train_images
    !unzip -q -n "$DATASET_DIR/stage_2_train_images.zip" -d /content/rsna_train_images
if not os.path.isdir(IMG_DIR):
    # Some ZIPs extract the .dcm files without the stage_2_train_images/ folder
    IMG_DIR = '/content/rsna_train_images'
    os.environ['IMG_DIR'] = IMG_DIR
print(IMG_DIR, '->', len([f for f in os.listdir(IMG_DIR) if f.endswith('.dcm')]), 'DICOM files')

### Step 5: Check inputs from Members 1 and 2
Needs `all_annotations.csv`, `train.csv`, `val.csv`, `test.csv` (Member 1) and `best_model.pth` (Member 2).

In [ ]:
for f in ['all_annotations.csv', 'train.csv', 'val.csv', 'test.csv']:
    p = f'{METADATA_DIR}/{f}'
    print(f'{f:22s}', 'OK' if os.path.exists(p) else 'MISSING', p)
print(f'{"best_model.pth":22s}', 'OK' if os.path.exists(CHECKPOINT) else 'MISSING — ask Member 2 for the checkpoint', CHECKPOINT)
if os.path.exists(CHECKPOINT):
    import torch
    ck = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
    if isinstance(ck, dict):
        print('  checkpoint epoch:', ck.get('epoch'), '| val AUC:', ck.get('val_auc'))

### Step 6: Grad-CAM on a single positive X-ray

In [ ]:
import pandas as pd
from IPython.display import Image, display

ann = pd.read_csv(f'{METADATA_DIR}/all_annotations.csv')
val = pd.read_csv(f'{METADATA_DIR}/val.csv')
pid = val[val.Target == 1].patientId.iloc[0]

!python rc/src/gradcam.py --image "$IMG_DIR/{pid}.dcm" --checkpoint "$CHECKPOINT" --output /content/gradcam_example.png
display(Image('/content/gradcam_example.png'))

### Step 7: Localization on `val.csv` — compare methods and thresholds
Evaluates every ground-truth-positive validation patient for each configuration
(`norm5` = original method; `denseblock3` = finer 14x14 heatmap; `+tta` = flip averaging;
`+prior` = train-set location prior; `+split` = one box per lung) at heatmap thresholds 0.2–0.8.
Also reports the selected method on the pneumonia cases the classifier detects (p ≥ 0.5).
Reports mask IoU, mean best IoU per GT box, recall/precision at IoU ≥ 0.3/0.5, the pointing game,
two image-free references, and a coordinate check (image sizes, GT boxes outside the image).

In [ ]:
!python rc/src/localization.py \
    --split_csv "$METADATA_DIR/val.csv" \
    --annotations_csv "$METADATA_DIR/all_annotations.csv" \
    --train_csv "$METADATA_DIR/train.csv" \
    --img_dir "$IMG_DIR" \
    --checkpoint "$CHECKPOINT" \
    --results_dir "$RESULTS_DIR"

In [ ]:
import json
val_report = json.load(open(f'{RESULTS_DIR}/localization_val_summary.json'))
BEST_CONFIG, BEST_T = val_report['best']['config'], val_report['best']['threshold']
print('Selected on validation set:', BEST_CONFIG, '@ threshold', BEST_T)
print(json.dumps({k: val_report[k] for k in ['coordinate_check', 'best', 'baseline', 'image_free_references']}, indent=2))
display(pd.read_csv(f'{RESULTS_DIR}/localization_val_comparison.csv')
        .sort_values('mean_mask_iou', ascending=False).head(15).round(3))
display(Image(f'{RESULTS_DIR}/localization_val_threshold_sweep.png'))
display(Image(f'{RESULTS_DIR}/localization_val_examples.png'))

### Step 8: Final localization result on held-out `test.csv`
Runs once, with the configuration and threshold selected on val (plus the original method for comparison).

In [ ]:
!python rc/src/localization.py \
    --split_csv "$METADATA_DIR/test.csv" \
    --annotations_csv "$METADATA_DIR/all_annotations.csv" \
    --train_csv "$METADATA_DIR/train.csv" \
    --img_dir "$IMG_DIR" \
    --checkpoint "$CHECKPOINT" \
    --results_dir "$RESULTS_DIR" \
    --use_val_selection "$RESULTS_DIR/localization_val_summary.json"

In [ ]:
test_report = json.load(open(f'{RESULTS_DIR}/localization_test_summary.json'))
print(json.dumps({k: test_report[k] for k in ['coordinate_check', 'best', 'baseline', 'image_free_references']}, indent=2))
display(Image(f'{RESULTS_DIR}/localization_test_examples.png'))

### Step 9: Live Streamlit demo
Starts the app in the background and opens a public `trycloudflare.com` tunnel. Open the printed URL,
then upload a `.dcm` file (download one from the extracted images below, or from the dataset folder).

In [ ]:
import subprocess, time, re

os.environ['PNEUMONIA_CHECKPOINT'] = CHECKPOINT
os.environ['PNEUMONIA_ANNOTATIONS'] = f'{METADATA_DIR}/all_annotations.csv'
os.environ['PNEUMONIA_CAM_THRESHOLD'] = str(BEST_T)
os.environ['PNEUMONIA_LOC_CONFIG'] = BEST_CONFIG

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

app = subprocess.Popen(['streamlit', 'run', 'rc/app/app.py', '--server.port', '8501', '--server.headless', 'true'],
                       stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if m:
        print('Demo URL:', m.group(0))
        break

In [ ]:
# Copy a few positive / negative test DICOMs somewhere easy to download for the demo
from google.colab import files
test = pd.read_csv(f'{METADATA_DIR}/test.csv')
demo_ids = list(test[test.Target == 1].patientId[:2]) + list(test[test.Target == 0].patientId[:1])
for pid in demo_ids:
    files.download(f'{IMG_DIR}/{pid}.dcm')

In [ ]:
# Stop the demo
app.terminate(); tunnel.terminate()